# Few-Shot Learning on Omniglot
Reimplements **5-way 1-shot** classification with:
- **Matching Networks** — Vinyals et al., NeurIPS 2016

Target scores from Table 1 of *Relation Network* (Sung et al., arXiv 1711.06025)

## 1. Install dependencies

In [ ]:
!pip install torch torchvision wandb numpy Pillow -q

## 2. Imports & reproducibility

In [ ]:
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import Omniglot
import wandb

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## 3. Configuration
All hyperparameters in one place. Change `MODEL` to `"matching"` to switch methods.

In [ ]:
# ── Task ──────────────────────────────────────────────────────────────────
MODEL       = "matching"   # "proto" | "matching"
N_WAY       = 5         # test-time way
K_SHOT      = 1         # support shots
N_QUERY     = 15         # query examples per class per episode

# Proto paper trains at higher way than test (Section 2.6).
# Set equal to N_WAY for Matching Nets.
TRAIN_N_WAY = 60 if MODEL == "proto" else N_WAY

# ── Data ───────────────────────────────────────────────────────────────────
DATA_DIR    = "./data"
IMAGE_SIZE  = 28

# ── Training ───────────────────────────────────────────────────────────────
EPOCHS           = 200    # 200 epochs × 1000 eps = 200k total episodes
TRAIN_EPISODES   = 1000   # episodes sampled per epoch
TEST_EPISODES    = 1000   # paper evaluates over 1000 episodes
LR               = 1e-3
LR_STEP_EPISODES = 2000   # halve LR every N episodes (Snell et al.)
EVAL_FREQ        = 5      # evaluate on test set every N epochs
HIDDEN_SIZE      = 64     # conv filters in embedding net
CKPT_DIR         = "./checkpoints"

# ── WandB ──────────────────────────────────────────────────────────────────
WANDB_PROJECT = "few-shot-omniglot"

print(f"Model      : {MODEL}")
print(f"Train ep   : {TRAIN_N_WAY}-way {K_SHOT}-shot")
print(f"Test  ep   : {N_WAY}-way {K_SHOT}-shot")
print(f"LR halve   : every {LR_STEP_EPISODES} episodes")

Model      : matching
Train ep   : 5-way 1-shot
Test  ep   : 5-way 1-shot
LR halve   : every 2000 episodes


## 4. Omniglot episodic dataset

Three paper-specific details baked in:
1. **Images inverted** — Omniglot is black-on-white; we flip so foreground = white
2. **Rotation augmentation** — each character class is replicated at 0°/90°/180°/270°,
   creating 4× more *classes* (not more samples per class). Background split: 964 × 4 = **3856 virtual classes**
3. **Episode sampling** — each `__getitem__` returns a full (support, query) episode

In [ ]:
class OmniglotFewShot(Dataset):
    """
    Episodic Omniglot dataset with rotation-augmented classes.
    Each __getitem__ returns one complete (support + query) episode.
    """

    def __init__(self, root, split="train", n_way=5, k_shot=1,
                 n_query=15, n_episodes=1000, image_size=28):
        self.n_way      = n_way
        self.k_shot     = k_shot
        self.n_query    = n_query
        self.n_episodes = n_episodes

        background = (split == "train")
        transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Lambda(lambda x: 1.0 - x),  # invert: foreground = white
        ])
        dataset = Omniglot(root=root, background=background,
                           transform=transform, download=True)

        # Group images by base class
        base = {}
        for img, label in dataset:
            base.setdefault(label, []).append(img)

        # Create 4 virtual classes per base class via rotation augmentation.
        # Each rotated version is a NEW class, not extra samples in the original.
        # (Vinyals et al. and Snell et al. both use this exact procedure.)
        self.class_to_imgs = {}
        virtual_label = 0
        for imgs in base.values():
            for degrees in [0, 90, 180, 270]:
                self.class_to_imgs[virtual_label] = [
                    transforms.functional.rotate(img, degrees) for img in imgs
                ]
                virtual_label += 1

        self.classes = list(self.class_to_imgs.keys())
        print(f"  [{split}] {len(base)} base classes × 4 rotations "
              f"= {len(self.classes)} virtual classes")
        assert len(self.classes) >= n_way

    def __len__(self):
        return self.n_episodes

    def __getitem__(self, _):
        episode_classes = random.sample(self.classes, self.n_way)
        support_imgs, support_lbls = [], []
        query_imgs,   query_lbls   = [], []

        for local_label, cls in enumerate(episode_classes):
            imgs = random.sample(self.class_to_imgs[cls], self.k_shot + self.n_query)
            support_imgs.extend(imgs[:self.k_shot])
            support_lbls.extend([local_label] * self.k_shot)
            query_imgs.extend(imgs[self.k_shot:])
            query_lbls.extend([local_label] * self.n_query)

        return (
            torch.stack(support_imgs),
            torch.tensor(support_lbls, dtype=torch.long),
            torch.stack(query_imgs),
            torch.tensor(query_lbls,   dtype=torch.long),
        )


# Build datasets
print("Loading datasets...")
train_dataset = OmniglotFewShot(DATA_DIR, split="train", n_way=TRAIN_N_WAY,
                                k_shot=K_SHOT, n_query=N_QUERY,
                                n_episodes=TRAIN_EPISODES, image_size=IMAGE_SIZE)
test_dataset  = OmniglotFewShot(DATA_DIR, split="test",  n_way=N_WAY,
                                k_shot=K_SHOT, n_query=N_QUERY,
                                n_episodes=TEST_EPISODES, image_size=IMAGE_SIZE)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False, num_workers=2)
print("Done.")

Loading datasets...


100%|██████████| 9.46M/9.46M [00:00<00:00, 135MB/s]


  [train] 964 base classes × 4 rotations = 3856 virtual classes


100%|██████████| 6.46M/6.46M [00:00<00:00, 122MB/s]


  [test] 659 base classes × 4 rotations = 2636 virtual classes
Done.


## 5. Model definitions

### Shared backbone
Both papers use the same 4-block CNN:
`Conv(64, 3×3) → BN → ReLU → MaxPool(2)` × 4 → 64-d embedding for 28×28 input

In [ ]:
def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class EmbeddingNet(nn.Module):
    """4-layer CNN backbone shared by both models. Output: (B, 64)."""
    def __init__(self, hidden_size=64):
        super().__init__()
        self.encoder = nn.Sequential(
            conv_block(1, hidden_size),
            conv_block(hidden_size, hidden_size),
            conv_block(hidden_size, hidden_size),
            conv_block(hidden_size, hidden_size),
        )

    def forward(self, x):
        return self.encoder(x).view(x.size(0), -1)  # (B, 64)

### Prototypical Networks (Snell et al., 2017)
Classify by nearest prototype under **squared Euclidean distance** (Eq. 1–2 of paper).
> Snell et al. show Euclidean distance substantially outperforms cosine for ProtoNets.

In [ ]:
class PrototypicalNet(nn.Module):
    """
    Snell et al., NeurIPS 2017.
    https://papers.nips.cc/paper_files/paper/2017/file/cb8da6767461f2812ae4290eac7cbc42-Paper.pdf
    """
    def __init__(self, hidden_size=64):
        super().__init__()
        self.encoder = EmbeddingNet(hidden_size)

    def forward(self, support_images, support_labels, query_images, n_way):
        z_support = self.encoder(support_images)  # (N*K, D)
        z_query   = self.encoder(query_images)    # (N*Q, D)

        # Prototype = mean embedding per class (Eq. 1)
        prototypes = torch.stack(
            [z_support[support_labels == c].mean(0) for c in range(n_way)]
        )  # (N, D)

        # Classify by nearest prototype under squared Euclidean distance (Eq. 2)
        dists = torch.cdist(z_query, prototypes)  # (N*Q, N)
        return F.log_softmax(-dists, dim=1)

### Matching Networks (Vinyals et al., 2016)
Classify via **cosine-softmax attention** over support embeddings (Eq. 1, Section 2.1.1).
This is the simple non-FCE variant (no bidirectional LSTM context embedding).

In [ ]:
class MatchingNet(nn.Module):
    """
    Vinyals et al., NeurIPS 2016 — simple (non-FCE) variant.
    https://papers.nips.cc/paper_files/paper/2016/file/90e1357833654983612fb05e3ec9148c-Paper.pdf
    """
    def __init__(self, hidden_size=64):
        super().__init__()
        self.encoder = EmbeddingNet(hidden_size)

    def forward(self, support_images, support_labels, query_images, n_way):
        z_support = self.encoder(support_images)  # (N*K, D)
        z_query   = self.encoder(query_images)    # (N*Q, D)

        # Cosine-softmax attention kernel (Section 2.1.1)
        z_s = F.normalize(z_support, dim=1)
        z_q = F.normalize(z_query,   dim=1)
        attn = F.softmax(z_q @ z_s.T, dim=1)  # (N*Q, N*K)

        # Weighted sum of one-hot labels → class scores (Eq. 1)
        one_hot = F.one_hot(support_labels, num_classes=n_way).float()  # (N*K, N)
        logits  = attn @ one_hot                                         # (N*Q, N)
        return torch.log(logits.clamp(min=1e-8))

## 6. Instantiate model & optimiser

In [ ]:
ModelCls = {"proto": PrototypicalNet, "matching": MatchingNet}[MODEL]
model     = ModelCls(hidden_size=HIDDEN_SIZE).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model  : {MODEL}")
print(f"Params : {n_params:,}")

Model  : matching
Params : 111,936


## 7. Episode runner & evaluation

In [ ]:
def run_episode(model, support_imgs, support_lbls, query_imgs, query_lbls,
                n_way, optimizer=None):
    """Single forward pass. Backprops if optimizer is given. Returns (loss, acc)."""
    log_p = model(support_imgs, support_lbls, query_imgs, n_way)
    loss  = F.nll_loss(log_p, query_lbls)
    acc   = (log_p.argmax(1) == query_lbls).float().mean().item()

    if optimizer is not None:
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return loss.item(), acc


def evaluate(model, loader, device, n_way, desc="test"):
    """Evaluate over all episodes in loader. Returns (mean_loss, mean_acc, 95%_CI)."""
    model.eval()
    losses, accs = [], []
    with torch.no_grad():
        for support_imgs, support_lbls, query_imgs, query_lbls in loader:
            support_imgs = support_imgs.squeeze(0).to(device)
            support_lbls = support_lbls.squeeze(0).to(device)
            query_imgs   = query_imgs.squeeze(0).to(device)
            query_lbls   = query_lbls.squeeze(0).to(device)
            loss, acc = run_episode(model, support_imgs, support_lbls,
                                    query_imgs, query_lbls, n_way)
            losses.append(loss)
            accs.append(acc)

    mean_acc  = np.mean(accs)
    ci_95     = 1.96 * np.std(accs) / np.sqrt(len(accs))
    mean_loss = np.mean(losses)
    print(f"  [{desc}] loss={mean_loss:.4f}  acc={mean_acc*100:.2f}% ± {ci_95*100:.2f}%")
    return mean_loss, mean_acc, ci_95

## 8. Initialise WandB
Run `wandb login` in a terminal first, or set `WANDB_API_KEY` as an environment variable.

In [ ]:
import os
os.environ["WANDB_API_KEY"] = WANDB_KEY  # paste your key directly

wandb.init(
    project=WANDB_PROJECT,
    name=f"{MODEL}_{N_WAY}way_{K_SHOT}shot",
    config=dict(
        model=MODEL, n_way=N_WAY, k_shot=K_SHOT, n_query=N_QUERY,
        train_n_way=TRAIN_N_WAY, hidden_size=HIDDEN_SIZE,
        lr=LR, lr_step_episodes=LR_STEP_EPISODES,
        epochs=EPOCHS, train_episodes=TRAIN_EPISODES, test_episodes=TEST_EPISODES,
    ),
)
print("WandB run:", wandb.run.name)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rm7569 (sinha-anushka12-na) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


WandB run: matching_5way_1shot


## 9. Training loop

**LR schedule:** halve every `LR_STEP_EPISODES` episodes (Snell et al. use 2000).
With 1000 episodes/epoch this fires at epoch 2, 4, 6, … — much finer-grained than epoch-level stepping.

In [ ]:
Path(CKPT_DIR).mkdir(parents=True, exist_ok=True)

best_acc    = 0.0
episode_idx = 0
current_lr  = LR

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_losses, epoch_accs = [], []

    for support_imgs, support_lbls, query_imgs, query_lbls in train_loader:
        support_imgs = support_imgs.squeeze(0).to(DEVICE)
        support_lbls = support_lbls.squeeze(0).to(DEVICE)
        query_imgs   = query_imgs.squeeze(0).to(DEVICE)
        query_lbls   = query_lbls.squeeze(0).to(DEVICE)

        loss, acc = run_episode(model, support_imgs, support_lbls,
                                query_imgs, query_lbls, TRAIN_N_WAY, optimizer)
        epoch_losses.append(loss)
        epoch_accs.append(acc)

        episode_idx += 1
        # Halve LR every LR_STEP_EPISODES episodes (paper: every 2000)
        if episode_idx % LR_STEP_EPISODES == 0:
            current_lr *= 0.5
            for pg in optimizer.param_groups:
                pg["lr"] = current_lr
            print(f"  [ep {episode_idx}] LR → {current_lr:.2e}")

    train_loss = np.mean(epoch_losses)
    train_acc  = np.mean(epoch_accs)

    if epoch % EVAL_FREQ == 0:
        print(f"\nEpoch {epoch}/{EPOCHS}  ep={episode_idx}  lr={current_lr:.2e}")
        print(f"  [train] loss={train_loss:.4f}  acc={train_acc*100:.2f}%")
        val_loss, val_acc, val_ci = evaluate(model, test_loader, DEVICE, N_WAY)

        wandb.log({
            "epoch": epoch, "episode": episode_idx,
            "train/loss": train_loss, "train/acc": train_acc,
            "test/loss": val_loss,   "test/acc": val_acc,
            "test/acc_ci_95": val_ci, "lr": current_lr,
        })

        if val_acc > best_acc:
            best_acc = val_acc
            ckpt_path = f"{CKPT_DIR}/best_{MODEL}.pt"
            torch.save(model.state_dict(), ckpt_path)
            print(f"  ✓ New best ({best_acc*100:.2f}%) → {ckpt_path}")
    else:
        wandb.log({
            "epoch": epoch, "episode": episode_idx,
            "train/loss": train_loss, "train/acc": train_acc,
            "lr": current_lr,
        })

print("\nTraining complete.")

  [ep 2000] LR → 5.00e-04
  [ep 4000] LR → 2.50e-04

Epoch 5/200  ep=5000  lr=2.50e-04
  [train] loss=1.1193  acc=89.90%
  [test] loss=1.2015  acc=84.39% ± 0.62%
  ✓ New best (84.39%) → ./checkpoints/best_matching.pt
  [ep 6000] LR → 1.25e-04
  [ep 8000] LR → 6.25e-05
  [ep 10000] LR → 3.13e-05

Epoch 10/200  ep=10000  lr=3.13e-05
  [train] loss=1.0975  acc=92.05%
  [test] loss=1.1986  acc=86.43% ± 0.60%
  ✓ New best (86.43%) → ./checkpoints/best_matching.pt
  [ep 12000] LR → 1.56e-05
  [ep 14000] LR → 7.81e-06

Epoch 15/200  ep=15000  lr=7.81e-06
  [train] loss=1.0932  acc=92.27%
  [test] loss=1.1888  acc=86.62% ± 0.58%
  ✓ New best (86.62%) → ./checkpoints/best_matching.pt
  [ep 16000] LR → 3.91e-06
  [ep 18000] LR → 1.95e-06
  [ep 20000] LR → 9.77e-07

Epoch 20/200  ep=20000  lr=9.77e-07
  [train] loss=1.0893  acc=92.64%
  [test] loss=1.1761  acc=87.37% ± 0.57%
  ✓ New best (87.37%) → ./checkpoints/best_matching.pt
  [ep 22000] LR → 4.88e-07
  [ep 24000] LR → 2.44e-07

Epoch 25/200 